# 03 — Measurement Estimation

Purpose: Convert landmarks and user-input height into pixel-to-cm scale and produce cm-level **fit signals** (shoulder / hip widths, torso / leg / arm lengths).

> Circumferences (chest / waist / hip) are out of scope here.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys, json
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.pose_detection import PoseResult
from src.measurement import estimate_fit_signals

LANDMARK_DIR = ROOT / "data" / "processed" / "landmarks"
OUT_CSV = ROOT / "data" / "processed" / "measurements" / "fit_signals.csv"
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
# user_id -> height_cm mapping (PoC: hardcoded or CSV).
USER_HEIGHT_CM = {
    # "sample_user_01": 170.0,
}


In [ ]:
rows = []
for jf in sorted(LANDMARK_DIR.glob("*.json")):
    data = json.loads(jf.read_text())
    pose = PoseResult(
        landmarks_px=np.array(data["landmarks_px"], dtype=np.float32),
        landmarks_world=np.array(data["landmarks_world"], dtype=np.float32),
        image_width=data["width"],
        image_height=data["height"],
        detected=True,
    )
    user_id = jf.stem
    height = USER_HEIGHT_CM.get(user_id)
    if height is None:
        print(f"skip {user_id}: no height in USER_HEIGHT_CM")
        continue
    sig = estimate_fit_signals(pose, height)
    rows.append({"user_id": user_id, **sig.as_dict()})

df = pd.DataFrame(rows)
df

In [ ]:
df.to_csv(OUT_CSV, index=False)
print(f"wrote: {OUT_CSV}")